In [188]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler

import seaborn as sns
import matplotlib.pyplot as plt

In [189]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("../data/processed/airbnb_rio/total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

In [190]:
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        calculated_host_listings_count AS host_listings_count,
        property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification, 
        snapshot_month AS month,
        security_deposit,
        cleaning_fee,
        instant_bookable,
        number_of_reviews
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("selected_df")

## Initial Parsing

In [191]:
selected_df.show()

+-------------------+-------------------+-----------------+-------------------+-------------+---------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+----------------+-----------------+
|           latitude|          longitude|host_is_superhost|host_listings_count|property_type|      room_type|accommodates|bathrooms|bedrooms|beds|           amenities| price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|instant_bookable|number_of_reviews|
+-------------------+-------------------+-----------------+-------------------+-------------+---------------+------------+---------+--------+----+--------------------+------+-----------------------------+--------------------------------+-----+----------------+------------+----------------+-----------------+
|          -22.99195|          -43.43592|              0.0|              

Shape of the dataset:

In [192]:
# change this to pure SQL
print(f'Rows: {selected_df.count()}, Columns: {len(selected_df.columns)}')

Rows: 784121, Columns: 19


Checking for Null's:

In [193]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+----------------+-----------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|room_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|instant_bookable|number_of_reviews|
+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+----------------+-----------------+
|       0|        0|              385|                  0|            0|        0|           0|     1493|     775|2334|        0|    0|                            0|                         

As we can see there's a lot of missing values in the `security_deposit` and `cleaning_fee` columns.

Dropping them from the database seems to be a good decision.

In [194]:
rows_before_drop = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude, longitude, host_is_superhost, host_listings_count,
        property_type, room_type, accommodates, bathrooms, bedrooms, beds,
        amenities, price, require_guest_profile_picture,
        require_guest_phone_verification, month,
        CASE WHEN security_deposit IS NOT NULL AND security_deposit > 0 THEN 1 ELSE 0 END AS has_security_deposit,
        instant_bookable, number_of_reviews
    FROM selected_df
""")

In [195]:
selected_df = selected_df.dropna()
print(f"({selected_df.count()}, {len(selected_df.columns)}) - {rows_before_drop - selected_df.count()} rows dropped")

(780036, 18) - 4085 rows dropped


Checking if there is any missing left:

In [196]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()


+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+--------------------+----------------+-----------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|room_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|has_security_deposit|instant_bookable|number_of_reviews|
+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+--------------------+----------------+-----------------+
|       0|        0|                0|                  0|            0|        0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|             

In [197]:
selected_df.createOrReplaceTempView('selected_df')

### Checking data types

In [198]:
selected_df.printSchema()

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- host_is_superhost: double (nullable = true)
 |-- host_listings_count: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: double (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- amenities: string (nullable = true)
 |-- price: double (nullable = true)
 |-- require_guest_profile_picture: double (nullable = true)
 |-- require_guest_phone_verification: double (nullable = true)
 |-- month: double (nullable = true)
 |-- has_security_deposit: integer (nullable = false)
 |-- instant_bookable: double (nullable = true)
 |-- number_of_reviews: double (nullable = true)



TODO: A brief explanation of what need to be changed

### Checking data types

In [199]:
selected_df = spark.sql("""
    SELECT
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        TRY_CAST(host_is_superhost AS BOOLEAN) AS host_is_superhost,
        TRY_CAST(TRY_CAST(host_listings_count AS DOUBLE) AS INT) AS host_listings_count,
        property_type,
        room_type,
        TRY_CAST(TRY_CAST(accommodates AS DOUBLE) AS INT) AS accommodates,
        TRY_CAST(TRY_CAST(bathrooms AS DOUBLE) AS INT) AS bathrooms,
        TRY_CAST(TRY_CAST(bedrooms AS DOUBLE) AS INT) AS bedrooms,
        TRY_CAST(TRY_CAST(beds AS DOUBLE) AS INT) AS beds,
        amenities,
        TRY_CAST(REGEXP_REPLACE(price, '[\\\\$,]', '') AS DOUBLE) AS price,
        TRY_CAST(require_guest_profile_picture AS BOOLEAN) AS require_guest_profile_picture,
        TRY_CAST(require_guest_phone_verification AS BOOLEAN) AS require_guest_phone_verification, 
        TRY_CAST(month AS INT) AS month,
        TRY_CAST(TRY_CAST(instant_bookable AS DOUBLE) AS INT) AS instant_bookable,
        TRY_CAST(TRY_CAST(number_of_reviews AS DOUBLE) AS INT) AS number_of_reviews,
        has_security_deposit
    FROM selected_df
""")

selected_df.createOrReplaceTempView("selected_df")

null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+-----------------+--------------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|room_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|instant_bookable|number_of_reviews|has_security_deposit|
+--------+---------+-----------------+-------------------+-------------+---------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+-----------------+--------------------+
|       0|        0|                0|                  0|            0|        0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|             

## EDA + Data Cleaning

### Dealing with outliers

In [200]:
def get_max_fence(column):
    qt = selected_df.approxQuantile(column, [0.25,0.75], 0.01)
    q1 = qt[0]
    upper = qt[1]
    iqr = upper - q1
    max_fence = upper + 1.5 * iqr
    return max_fence

#### `host_listing_count`

In [201]:
listing_count_max_fence = get_max_fence('host_listings_count')
print(listing_count_max_fence)

3.5


In [202]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE host_listings_count <= {listing_count_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

148367 rows were deleted.


In [203]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        CASE WHEN host_listings_count = 0.0 THEN 1.0 ELSE host_listings_count END AS host_listings_count,
        property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        has_security_deposit,
        instant_bookable,
        number_of_reviews
    FROM selected_df
""")

#### `price`

In [204]:
price_max_fence = get_max_fence('price')
print(price_max_fence)

1274.5


In [205]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE price <= {price_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

63917 rows were deleted.


#### `property_type`

In [206]:
categories_to_append = ('Aparthotel', 'Earth house', 'Chalet', 'Cottage', 'Tiny house',
                        'Boutique hotel', 'Hotel', 'Casa particular (Cuba)', 'Bungalow',
                        'Nature lodge', 'Cabin', 'Castle', 'Treehouse', 'Island', 'Boat', 'Tent',
                        'Resort', 'Hut', 'Campsite', 'Barn', 'Dorm', 'Camper/RV', 'Farm stay', 'Yurt',
                        'Tipi', 'Pension (South Korea)', 'Dome house', 'Igloo', 'Casa particular',
                        'Houseboat', 'Lighthouse', 'Plane', 'Train', 'Parking Space')

category_list_sql = ", ".join([f"'" + c.replace("'", "''") + "'" for c in categories_to_append])
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        CASE WHEN property_type IN ({category_list_sql}) THEN 'Other' ELSE property_type END AS property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        has_security_deposit,
        instant_bookable,
        number_of_reviews
    FROM selected_df
""")

#### `beds`

In [207]:
beds_max_fence = get_max_fence('beds')
print(beds_max_fence)

6.0


In [208]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE beds <= {beds_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

11698 rows were deleted.


#### `amenities`

In [209]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        has_security_deposit,
        instant_bookable,
        number_of_reviews,
        SIZE(SPLIT(amenities, ',')) + 1 AS n_amenities
    FROM selected_df
""")
selected_df.createOrReplaceTempView('selected_df')
mode_value = spark.sql("""
    SELECT n_amenities
    FROM selected_df
    WHERE amenities <> '{}'
    GROUP BY n_amenities
    ORDER BY COUNT(*) DESC
    LIMIT 1
""").first()['n_amenities']
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        has_security_deposit,
        instant_bookable,
        number_of_reviews,
        CASE WHEN amenities = '{{}}' THEN {mode_value} ELSE n_amenities END AS n_amenities
    FROM selected_df
""")

In [210]:
amenities_max_fence = get_max_fence('n_amenities')
print(amenities_max_fence)

38.0


In [211]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE n_amenities <= {amenities_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

26744 rows were deleted.


## Encoding

In [212]:
selected_df.createOrReplaceTempView('selected_df')
df_label_encoder = spark.sql("""
    WITH property_type_index AS (
        SELECT
            property_type,
            ROW_NUMBER() OVER (ORDER BY cnt DESC, property_type ASC) - 1 AS property_type_encoded
        FROM (
            SELECT property_type, COUNT(*) AS cnt
            FROM selected_df
            GROUP BY property_type
        )
    )
    SELECT
        s.latitude,
        s.longitude,
        CASE
            WHEN s.host_is_superhost IS TRUE THEN 1
            WHEN s.host_is_superhost IS FALSE THEN 0
            ELSE CAST(s.host_is_superhost AS INT)
        END AS host_is_superhost,
        s.host_listings_count,
        p.property_type_encoded AS property_type,
        s.accommodates,
        s.bathrooms,
        s.bedrooms,
        s.beds,
        s.price,
        CASE
            WHEN s.require_guest_profile_picture IS TRUE THEN 1
            WHEN s.require_guest_profile_picture IS FALSE THEN 0
            ELSE CAST(s.require_guest_profile_picture AS INT)
        END AS require_guest_profile_picture,
        CASE
            WHEN s.require_guest_phone_verification IS TRUE THEN 1
            WHEN s.require_guest_phone_verification IS FALSE THEN 0
            ELSE CAST(s.require_guest_phone_verification AS INT)
        END AS require_guest_phone_verification,
        s.month,
        s.n_amenities,
        CASE WHEN s.room_type = 'Private room' THEN 1 ELSE 0 END AS room_type_private,
        CASE WHEN s.room_type = 'Shared room' THEN 1 ELSE 0 END AS room_type_shared,
        CASE WHEN s.room_type = 'Hotel room' THEN 1 ELSE 0 END AS room_type_hotel,
        s.instant_bookable,
        s.number_of_reviews,
        s.has_security_deposit
    FROM selected_df s
    LEFT JOIN property_type_index p
        ON s.property_type = p.property_type
""")
print('Columns encoded')

Columns encoded


## Model

In [213]:
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=100,
            regParam=0.01,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            seed=SEED,
            maxDepth=10,
            maxBins=64
        )
    ),
]

In [214]:
feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month",
    "property_type",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "n_amenities",
    "room_type_private",
    "room_type_shared",
    "room_type_hotel",
    "instant_bookable",
    "number_of_reviews",
    "has_security_deposit"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(df_label_encoder)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

print(f"Linhas no Treino: {train_prepared.count()}")
print(f"Linhas no Teste: {test_prepared.count()}")

Linhas no Treino: 423648


Linhas no Teste: 105662


In [215]:
def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [216]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)

RMSE: 238.92
MAE: 180.12
R2: 0.317050


RMSE: 209.31
MAE: 150.08
R2: 0.475814
